# Motion Parallax Analysis: Virtual Rodent Gap-Crossing

Replicates key analyses from **Parker et al. (eLife 2022)** for the virtual rodent gap-crossing task:

1. **Psychometric curves** — success rate vs gap distance
2. **Head kinematics** — vertical head movements (motion parallax), head pitch (position parallax)
3. **Approach behavior** — duration, distance traveled during gap approach
4. **Condition comparisons** — binocular vs monocular

**Data sources** (in priority order):
1. **Inline rollout** — generates fresh data by loading the checkpoint and running the agent (default)
2. **Pre-collected `.npz`** — loads from `../outputs/motion_parallax/` if available and `GENERATE_DATA = False`

In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.5"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

%matplotlib inline

import gc
import json

import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from pathlib import Path
from scipy import stats
from IPython.display import Video, display
import vnl_playground.naccdmax_patch

# Analysis modules
from vnl_playground.tasks.rodent.analysis.head_kinematics import (
    HeadPose, extract_head_pose, detect_approach_windows, ApproachWindow,
)
from vnl_playground.tasks.rodent.analysis.motion_parallax_analysis import (
    count_vertical_movements, compute_movement_amplitude,
    compute_head_pitch_stats, compute_total_head_distance, compute_approach_duration,
)
from vnl_playground.tasks.rodent.analysis.run_gap_psychometrics import (
    compute_gap_metrics, bin_metrics_by_gap_distance, compare_conditions,
    plot_psychometric_curves, plot_head_movement_comparison,
    plot_metric_vs_gap_distance, plot_parker_comparison_panel,
)
# Rollout infrastructure
from vnl_playground.tasks.rodent.analysis.collect_run_gap_data import (
    setup_env_and_policy,
    load_config,
    resolve_body_and_camera_ids,
    apply_monocular_mask,
)
from vnl_playground.tasks.rodent.vision_jax import JaxVisionRenderer, VisionRenderWrapper
from mujoco.mjx.warp.types import DATA_NON_VMAP

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
GENERATE_DATA = True  # Set False to skip rollout and load from npz

CHECKPOINT_DIR = Path("/home/scott/SalkResearch/data/bino_run_gaps/260307_110240")
PRIOR_DIR = Path("/home/scott/SalkResearch/data/prior")

DATA_DIR = Path("../outputs/motion_parallax")
FIGURES_DIR = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = ["binocular", "monocular_left", "monocular_right"]
COLORS = {
    "binocular": "tab:blue",
    "monocular_left": "tab:pink",
    "monocular_right": "tab:red",
}

# Rollout params
N_EPISODES = 20  # episodes per condition for inline rollout
MAX_STEPS = 2000  # must match episode_length from training
CTRL_DT = 0.01  # 100 Hz control rate
SEED = 42

# Video params
VIDEO_FPS = 50
VIDEO_HEIGHT = 480
VIDEO_WIDTH = 640

plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 150,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
    }
)

print("Configuration OK")

## 1. Load Environment & Policy

In [ ]:
if GENERATE_DATA:
    ckpt_config = load_config(str(CHECKPOINT_DIR))
    ### OVERWRITE TO DEFAULT
    ckpt_config["env_config"]["env_args"]["aesthetic"] = "default"

    wrapped_env, policy_fn, params_tuple, mj_model, base_env = setup_env_and_policy(
        checkpoint_path=str(CHECKPOINT_DIR),
        prior_checkpoint_path=str(PRIOR_DIR),
        ckpt_config=ckpt_config,
        seed=SEED,
    )
    body_cam_ids = resolve_body_and_camera_ids(mj_model)

    # Binocular renderers (nworld=1 for single-env rollout)
    env_cfg = ckpt_config["env_config"]
    left_renderer = JaxVisionRenderer(
        mj_model=base_env.mj_model,
        mjx_model=base_env.mjx_model,
        nworld=1,
        camera_name=env_cfg.get("left_camera_name", "eye_left-rodent"),
        width=env_cfg.get("vision_width", 32),
        height=env_cfg.get("vision_height", 32),
        grayscale=env_cfg.get("grayscale", True),
        use_textures=env_cfg.get("use_textures", True),
        use_shadows=env_cfg.get("use_shadows", True),
    )
    right_renderer = JaxVisionRenderer(
        mj_model=base_env.mj_model,
        mjx_model=base_env.mjx_model,
        nworld=1,
        camera_name=env_cfg.get("right_camera_name", "eye_right-rodent"),
        width=env_cfg.get("vision_width", 32),
        height=env_cfg.get("vision_height", 32),
        grayscale=env_cfg.get("grayscale", True),
        use_textures=env_cfg.get("use_textures", True),
        use_shadows=env_cfg.get("use_shadows", True),
    )

    def _add_batch_dim_for_warp(data):
        """Add leading batch dim to Data, skipping non-vmap fields."""

        def _maybe_expand(path, x):
            parts = [p.name for p in path if hasattr(p, "name") and p.name != "_impl"]
            attr = "__".join(parts)
            if attr in DATA_NON_VMAP:
                return x
            return x[None, ...]

        return jax.tree.map_with_path(_maybe_expand, data)

    def render_binocular(data):
        data_b = _add_batch_dim_for_warp(data)
        left = left_renderer.render(data_b)[0]
        right = right_renderer.render(data_b)[0]
        return jnp.concatenate([left, right], axis=-1)

    @jax.jit
    def eval_reset(rng):
        state = wrapped_env.reset(rng)
        vision = render_binocular(state.data)
        return state.replace(obs=VisionRenderWrapper._inject_vision(state.obs, vision))

    @jax.jit
    def eval_step(state, action):
        state = wrapped_env.step(state, action)
        vision = render_binocular(state.data)
        return state.replace(obs=VisionRenderWrapper._inject_vision(state.obs, vision))

    # Third-person renderer for video
    mj_data = mujoco.MjData(mj_model)
    mj_renderer = mujoco.Renderer(mj_model, height=VIDEO_HEIGHT, width=VIDEO_WIDTH)

    camera = mujoco.MjvCamera()
    camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
    for name in ["torso-rodent", "torso"]:
        try:
            camera.trackbodyid = mj_model.body(name).id
            break
        except Exception:
            continue
    camera.distance = 1.0
    camera.azimuth = 90
    camera.elevation = -20
    camera.lookat[:] = [0, 0, 0.3]

    print("Environment, policy, and renderers ready.")
else:
    print("GENERATE_DATA=False — skipping env/policy setup.")

## 1b. Run Rollout & Collect Kinematics

Runs `N_EPISODES` single-env rollouts, recording per-timestep skull/torso kinematics
and gap geometry. First episode also records qpos + ego frames for the rollout video.

In [ ]:
if GENERATE_DATA:
    from vnl_playground.tasks.rodent.analysis.collect_run_gap_data import (
        _extract_gap_geometry_batch,
    )

    skull_id = body_cam_ids["skull_body_id"]
    torso_id = body_cam_ids["torso_body_id"]
    hand_l_id = mj_model.body("hand_L-rodent").id
    hand_r_id = mj_model.body("hand_R-rodent").id

    # Resolve touch sensor addresses for physics-based contact detection
    palm_l_adr = mj_model.sensor("palm_L-rodent").id
    palm_r_adr = mj_model.sensor("palm_R-rodent").id
    palm_l_sensoradr = mj_model.sensor_adr[palm_l_adr]
    palm_r_sensoradr = mj_model.sensor_adr[palm_r_adr]

    # ── Batched renderers (nworld=N_EPISODES) for parallel rollout ──
    env_cfg = ckpt_config["env_config"]
    _renderer_kwargs = dict(
        mj_model=base_env.mj_model,
        mjx_model=base_env.mjx_model,
        nworld=N_EPISODES,
        width=env_cfg.get("vision_width", 32),
        height=env_cfg.get("vision_height", 32),
        grayscale=env_cfg.get("grayscale", True),
        use_textures=env_cfg.get("use_textures", True),
        use_shadows=env_cfg.get("use_shadows", True),
    )
    left_renderer_batch = JaxVisionRenderer(
        camera_name=env_cfg.get("left_camera_name", "eye_left-rodent"),
        **_renderer_kwargs,
    )
    right_renderer_batch = JaxVisionRenderer(
        camera_name=env_cfg.get("right_camera_name", "eye_right-rodent"),
        **_renderer_kwargs,
    )

    # ── Vmapped reset/step with batched vision rendering ──
    vmapped_reset = jax.vmap(wrapped_env.reset)
    vmapped_step = jax.vmap(wrapped_env.step)

    def _render_and_inject_batch(state):
        """Render binocular vision for all envs and inject into obs."""
        left = left_renderer_batch.render(state.data)  # (N_EPISODES, H, W, C)
        right = right_renderer_batch.render(state.data)  # (N_EPISODES, H, W, C)
        vision = jnp.concatenate([left, right], axis=-1)  # (N_EPISODES, H, W, 2C)
        return state.replace(obs=VisionRenderWrapper._inject_vision(state.obs, vision))

    @jax.jit
    def batched_reset(rngs):
        state = vmapped_reset(rngs)
        return _render_and_inject_batch(state)

    @jax.jit
    def batched_step(state, actions):
        state = vmapped_step(state, actions)
        return _render_and_inject_batch(state)

    # ── Reset all N_EPISODES environments in parallel ──
    rng = jax.random.PRNGKey(SEED)
    rng, batch_rng, act_rng = jax.random.split(rng, 3)
    reset_rngs = jax.random.split(batch_rng, N_EPISODES)

    condition = "binocular"
    print(f"Running {N_EPISODES} episodes in parallel (condition={condition})...")
    state = batched_reset(reset_rngs)

    # Extract gap geometry from batched initial state
    gap_edges, gap_lens = _extract_gap_geometry_batch(base_env, state.data)
    # gap_edges, gap_lens: (N_EPISODES, n_platforms)

    # ── Pre-allocate per-timestep arrays: (T, N_EPISODES, ...) ──
    T = MAX_STEPS + 1
    batch_skull_xpos = np.zeros((T, N_EPISODES, 3), dtype=np.float32)
    batch_skull_xmat = np.zeros((T, N_EPISODES, 3, 3), dtype=np.float32)
    batch_torso_xpos = np.zeros((T, N_EPISODES, 3), dtype=np.float32)
    batch_torso_xmat = np.zeros((T, N_EPISODES, 3, 3), dtype=np.float32)
    batch_torso_linvel = np.zeros((T, N_EPISODES, 3), dtype=np.float32)
    batch_hand_l_xpos = np.zeros((T, N_EPISODES, 3), dtype=np.float32)
    batch_hand_r_xpos = np.zeros((T, N_EPISODES, 3), dtype=np.float32)
    batch_palm_l_touch = np.zeros((T, N_EPISODES), dtype=np.float32)
    batch_palm_r_touch = np.zeros((T, N_EPISODES), dtype=np.float32)
    batch_done = np.zeros((T, N_EPISODES), dtype=bool)

    def _record_batch(data, t_idx):
        xpos = np.asarray(data.xpos)  # (N_EPISODES, nbody, 3)
        xmat = np.asarray(data.xmat)  # (N_EPISODES, nbody, 3, 3)
        batch_skull_xpos[t_idx] = xpos[:, skull_id]
        batch_skull_xmat[t_idx] = xmat[:, skull_id]
        batch_torso_xpos[t_idx] = xpos[:, torso_id]
        batch_torso_xmat[t_idx] = xmat[:, torso_id]
        batch_torso_linvel[t_idx] = np.asarray(data._impl.subtree_linvel)[:, torso_id]
        batch_hand_l_xpos[t_idx] = xpos[:, hand_l_id]
        batch_hand_r_xpos[t_idx] = xpos[:, hand_r_id]
        # Record palm touch sensor values (normal force on palm sites)
        sdata = np.asarray(data.sensordata)  # (N_EPISODES, n_sensordata)
        batch_palm_l_touch[t_idx] = sdata[:, palm_l_sensoradr]
        batch_palm_r_touch[t_idx] = sdata[:, palm_r_sensoradr]

    # Record initial state (t=0)
    _record_batch(state.data, 0)

    # Video storage (episode 0): extract ego from the obs vision field
    n_vis_ch = state.obs["vision"].shape[-1] // 2  # channels per eye
    video_qpos = [np.array(state.data.qpos[0])]
    video_ego_left = [np.array(state.obs["vision"][0, :, :, :n_vis_ch])]
    video_ego_right = [np.array(state.obs["vision"][0, :, :, n_vis_ch:])]

    # ── Step loop: all episodes advance simultaneously ──
    env_done = np.zeros(N_EPISODES, dtype=bool)
    actual_steps = MAX_STEPS

    for t in range(MAX_STEPS):
        _, act_rng = jax.random.split(act_rng)
        actions, _ = policy_fn(params_tuple, state.obs, act_rng)
        state = batched_step(state, actions)

        _record_batch(state.data, t + 1)
        step_done = np.asarray(state.done) > 0.5  # (N_EPISODES,)
        batch_done[t + 1] = step_done

        # Video: record episode 0 up to and including its done frame
        if not env_done[0]:
            video_qpos.append(np.array(state.data.qpos[0]))
            video_ego_left.append(np.array(state.obs["vision"][0, :, :, :n_vis_ch]))
            video_ego_right.append(np.array(state.obs["vision"][0, :, :, n_vis_ch:]))

        env_done |= step_done

        if np.all(env_done):
            actual_steps = t + 1
            break

        if (t + 1) % 500 == 0:
            n_done = env_done.sum()
            print(f"  Step {t+1}/{MAX_STEPS}: {n_done}/{N_EPISODES} episodes done")

    # ── Segment per-episode data (trim to actual episode length) ──
    all_skull_xpos, all_skull_xmat = [], []
    all_torso_xpos, all_torso_xmat, all_torso_linvel = [], [], []
    all_hand_l_xpos, all_hand_r_xpos = [], []
    all_palm_l_touch, all_palm_r_touch = [], []
    all_episode_lengths = []
    all_gap_leading_edges, all_gap_lengths, all_gap_episode_ids = [], [], []

    for ep in range(N_EPISODES):
        done_flags = batch_done[1 : actual_steps + 1, ep]
        done_indices = np.where(done_flags)[0]
        if len(done_indices) > 0:
            ep_steps = int(done_indices[0]) + 1
        else:
            ep_steps = actual_steps
        ep_len = ep_steps + 1  # +1 for initial state

        all_skull_xpos.append(batch_skull_xpos[:ep_len, ep].copy())
        all_skull_xmat.append(batch_skull_xmat[:ep_len, ep].copy())
        all_torso_xpos.append(batch_torso_xpos[:ep_len, ep].copy())
        all_torso_xmat.append(batch_torso_xmat[:ep_len, ep].copy())
        all_torso_linvel.append(batch_torso_linvel[:ep_len, ep].copy())
        all_hand_l_xpos.append(batch_hand_l_xpos[:ep_len, ep].copy())
        all_hand_r_xpos.append(batch_hand_r_xpos[:ep_len, ep].copy())
        all_palm_l_touch.append(batch_palm_l_touch[:ep_len, ep].copy())
        all_palm_r_touch.append(batch_palm_r_touch[:ep_len, ep].copy())
        all_episode_lengths.append(ep_len)
        all_gap_leading_edges.append(gap_edges[ep])
        all_gap_lengths.append(gap_lens[ep])
        all_gap_episode_ids.append(np.full(len(gap_edges[ep]), ep, dtype=np.int32))

        max_x = batch_torso_xpos[:ep_len, ep, 0].max()
        if (ep + 1) % 5 == 0 or ep == 0:
            ep_done = len(done_indices) > 0
            print(
                f"  Episode {ep+1}/{N_EPISODES}: {ep_len} steps, max_x={max_x:.2f}m"
                + (" (done)" if ep_done else " (timeout)")
            )

    # Pack into npz-compatible dict (same format as before)
    inline_data = {
        "skull_xpos": np.concatenate(all_skull_xpos, axis=0),
        "skull_xmat": np.concatenate(all_skull_xmat, axis=0),
        "torso_xpos": np.concatenate(all_torso_xpos, axis=0),
        "torso_xmat": np.concatenate(all_torso_xmat, axis=0),
        "torso_linvel": np.concatenate(all_torso_linvel, axis=0),
        "hand_l_xpos": np.concatenate(all_hand_l_xpos, axis=0),
        "hand_r_xpos": np.concatenate(all_hand_r_xpos, axis=0),
        "palm_l_touch": np.concatenate(all_palm_l_touch, axis=0),
        "palm_r_touch": np.concatenate(all_palm_r_touch, axis=0),
        "episode_lengths": np.array(all_episode_lengths, dtype=np.int32),
        "n_episodes": N_EPISODES,
        "gap_leading_edges_flat": np.concatenate(all_gap_leading_edges, axis=0),
        "gap_lengths_flat": np.concatenate(all_gap_lengths, axis=0),
        "gap_episode_ids": np.concatenate(all_gap_episode_ids, axis=0),
    }

    total_ts = inline_data["skull_xpos"].shape[0]
    print(f"\nDone. {N_EPISODES} episodes, {total_ts} total timesteps.")
    print(f"Video frames recorded: {len(video_qpos)} (episode 0)")
else:
    inline_data = None
    print("Skipping rollout — will load from npz.")

## 1c. Rollout Video

Third-person tracking view with binocular egocentric vision overlay (left eye / right eye).
Use this to qualitatively verify the agent is running, crossing gaps, and the vision looks correct.

In [ ]:
if GENERATE_DATA and len(video_qpos) > 0:

    def render_third_person(qpos):
        mj_data.qpos = qpos
        mujoco.mj_forward(mj_model, mj_data)
        mj_renderer.update_scene(mj_data, camera)
        return mj_renderer.render().copy()

    def overlay_ego_binocular(frame, ego_left, ego_right, scale=3):
        """Overlay left and right eye images side-by-side on upper-left of frame."""
        out = frame.copy()
        pad = 4

        for i, (ego, label) in enumerate([(ego_left, "L"), (ego_right, "R")]):
            img = (np.clip(ego, 0, 1) * 255).astype(np.uint8)
            if img.ndim == 2:
                img = img[:, :, None]
            if img.shape[-1] == 1:
                img = np.repeat(img, 3, axis=-1)
            img = np.repeat(np.repeat(img, scale, axis=0), scale, axis=1)
            sh, sw = img.shape[:2]
            x0 = pad + i * (sw + pad + 2)
            y0 = pad
            y1, x1 = y0 + sh, x0 + sw
            if y1 + pad < out.shape[0] and x1 + pad < out.shape[1]:
                out[y0 - 1 : y1 + 1, x0 - 1 : x1 + 1] = 255  # border
                out[y0:y1, x0:x1] = img
        return out

    ROLLOUT_VIDEO_PATH = str(DATA_DIR / "rollout_binocular.mp4")

    fig_ro, ax_ro = plt.subplots(1, 1, figsize=(8, 6), dpi=100)
    im_ro = ax_ro.imshow(np.zeros((VIDEO_HEIGHT, VIDEO_WIDTH, 3), dtype=np.uint8))
    ax_ro.axis("off")
    title_ro = ax_ro.set_title("", fontsize=11, fontweight="bold")
    fig_ro.tight_layout()

    n_frames = len(video_qpos)
    print(f"Rendering rollout video ({n_frames} frames at {VIDEO_FPS} fps)...")
    vid_writer = FFMpegWriter(fps=VIDEO_FPS)
    with vid_writer.saving(fig_ro, ROLLOUT_VIDEO_PATH, dpi=100):
        for t in range(n_frames):
            frame = render_third_person(video_qpos[t])
            frame = overlay_ego_binocular(
                frame, video_ego_left[t], video_ego_right[t], scale=3
            )
            im_ro.set_data(frame)
            title_ro.set_text(f"Step {t}/{n_frames-1}")
            vid_writer.grab_frame()
            if (t + 1) % 200 == 0:
                print(f"  {t+1}/{n_frames} frames")

    plt.close(fig_ro)
    print(f"Video saved: {ROLLOUT_VIDEO_PATH}")
else:
    print("No video data — skipping.")

In [ ]:
if GENERATE_DATA and len(video_qpos) > 0:
    display(Video(ROLLOUT_VIDEO_PATH, embed=True, width=640))

## 1d. Rollout Filmstrip

Quick visual check: 10 evenly-spaced frames from the first episode with binocular ego overlays.

In [ ]:
if GENERATE_DATA and len(video_qpos) > 0:
    n_show = 10
    n_frames = len(video_qpos)
    show_steps = np.linspace(0, n_frames - 1, n_show, dtype=int)

    fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))

    for i, step in enumerate(show_steps):
        # Third-person with ego overlay
        frame = render_third_person(video_qpos[step])
        frame = overlay_ego_binocular(
            frame, video_ego_left[step], video_ego_right[step]
        )
        axes[0, i].imshow(frame)
        axes[0, i].set_title(f"Step {step}", fontsize=9)
        axes[0, i].axis("off")

        # Left eye
        ego_l = video_ego_left[step]
        axes[1, i].imshow(
            ego_l[:, :, 0] if ego_l.shape[-1] == 1 else ego_l,
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        axes[1, i].set_title("Left eye", fontsize=8)
        axes[1, i].axis("off")

        # Right eye
        ego_r = video_ego_right[step]
        axes[2, i].imshow(
            ego_r[:, :, 0] if ego_r.shape[-1] == 1 else ego_r,
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        axes[2, i].set_title("Right eye", fontsize=8)
        axes[2, i].axis("off")

    axes[0, 0].set_ylabel("3rd person", fontsize=10)
    axes[1, 0].set_ylabel("Left eye (32×32)", fontsize=10)
    axes[2, 0].set_ylabel("Right eye (32×32)", fontsize=10)
    plt.suptitle("Binocular Rollout Filmstrip (Episode 0)", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "rollout_filmstrip.png", dpi=150)
    plt.show()
else:
    print("No video data — skipping.")

## 1e. Monocular Validation Videos

Same agent, same seed (same gap layout), but with one eye blacked out. The ego overlay shows exactly what the policy receives — one eye active, one eye zeroed. Compare with the binocular video above to see behavioral differences under monocular deprivation.

In [ ]:
if GENERATE_DATA:
    monocular_conditions = ["monocular_left", "monocular_right"]

    for condition in monocular_conditions:
        print(f"\n{'='*60}")
        print(f"Recording {condition} video (same seed as binocular for comparison)...")

        # Use same seed so gap layout matches binocular video
        rng_v = jax.random.PRNGKey(SEED)
        rng_v, reset_rng, act_rng = jax.random.split(rng_v, 3)
        state = eval_reset(reset_rng)

        n_ch = state.obs["vision"].shape[-1] // 2
        vid_qpos = [np.array(state.data.qpos)]

        # Apply mask to initial obs vision for video display
        masked_vis = np.array(apply_monocular_mask(state.obs["vision"], condition))
        vid_ego_l = [masked_vis[:, :, :n_ch]]
        vid_ego_r = [masked_vis[:, :, n_ch:]]

        for t in range(MAX_STEPS):
            # Policy sees masked vision
            obs = state.obs
            obs = {**obs, "vision": apply_monocular_mask(obs["vision"], condition)}
            _, act_rng = jax.random.split(act_rng)
            action, _ = policy_fn(params_tuple, obs, act_rng)
            state = eval_step(state, action)

            vid_qpos.append(np.array(state.data.qpos))
            masked_vis = np.array(apply_monocular_mask(state.obs["vision"], condition))
            vid_ego_l.append(masked_vis[:, :, :n_ch])
            vid_ego_r.append(masked_vis[:, :, n_ch:])

            if float(state.done) > 0.5:
                break

        n_frames = len(vid_qpos)
        print(f"  {n_frames} frames recorded")

        # ── Render video ──
        video_path = str(DATA_DIR / f"rollout_{condition}.mp4")
        fig_v, ax_v = plt.subplots(1, 1, figsize=(8, 6), dpi=100)
        im_v = ax_v.imshow(np.zeros((VIDEO_HEIGHT, VIDEO_WIDTH, 3), dtype=np.uint8))
        ax_v.axis("off")
        title_v = ax_v.set_title("", fontsize=11, fontweight="bold")
        fig_v.tight_layout()

        print(f"  Rendering video ({n_frames} frames)...")
        writer = FFMpegWriter(fps=VIDEO_FPS)
        with writer.saving(fig_v, video_path, dpi=100):
            for t in range(n_frames):
                frame = render_third_person(vid_qpos[t])
                frame = overlay_ego_binocular(
                    frame, vid_ego_l[t], vid_ego_r[t], scale=3
                )
                im_v.set_data(frame)
                title_v.set_text(f"{condition}  Step {t}/{n_frames-1}")
                writer.grab_frame()
        plt.close(fig_v)
        print(f"  Saved: {video_path}")
        display(Video(video_path, embed=True, width=640))

        # ── Filmstrip ──
        n_show = 10
        show_steps = np.linspace(0, n_frames - 1, n_show, dtype=int)
        fig_fs, axes_fs = plt.subplots(3, n_show, figsize=(3 * n_show, 9))

        for i, step in enumerate(show_steps):
            frame = render_third_person(vid_qpos[step])
            frame = overlay_ego_binocular(frame, vid_ego_l[step], vid_ego_r[step])
            axes_fs[0, i].imshow(frame)
            axes_fs[0, i].set_title(f"Step {step}", fontsize=9)
            axes_fs[0, i].axis("off")

            ego_l = vid_ego_l[step]
            axes_fs[1, i].imshow(
                ego_l[:, :, 0] if ego_l.shape[-1] == 1 else ego_l,
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            axes_fs[1, i].set_title("Left eye", fontsize=8)
            axes_fs[1, i].axis("off")

            ego_r = vid_ego_r[step]
            axes_fs[2, i].imshow(
                ego_r[:, :, 0] if ego_r.shape[-1] == 1 else ego_r,
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            axes_fs[2, i].set_title("Right eye", fontsize=8)
            axes_fs[2, i].axis("off")

        axes_fs[0, 0].set_ylabel("3rd person", fontsize=10)
        axes_fs[1, 0].set_ylabel("Left eye (32x32)", fontsize=10)
        axes_fs[2, 0].set_ylabel("Right eye (32x32)", fontsize=10)
        plt.suptitle(
            f"{condition.replace('_', ' ').title()} Rollout Filmstrip", fontsize=14
        )
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"rollout_filmstrip_{condition}.png", dpi=150)
        plt.show()

    print("\nAll monocular videos done.")
else:
    print("Skipping — GENERATE_DATA=False")

In [ ]:
# Optionally save inline data to npz for reuse
SAVE_INLINE_DATA = True

if GENERATE_DATA and SAVE_INLINE_DATA and inline_data is not None:
    save_path = DATA_DIR / "run_gap_kinematics_binocular.npz"
    np.savez_compressed(
        save_path,
        **{
            k: (np.array(v) if not isinstance(v, np.ndarray) else v)
            for k, v in inline_data.items()
        },
    )
    print(
        f"Saved inline data to {save_path} ({save_path.stat().st_size / 1024:.0f} KB)"
    )

## 1e. Load & Segment Data for Analysis

Uses inline rollout data if available (`GENERATE_DATA=True`), otherwise loads from pre-collected `.npz` files.

In [ ]:
def load_kinematics_data(data_dir, condition):
    """Load pre-collected rollout kinematics from npz."""
    path = data_dir / f"run_gap_kinematics_{condition}.npz"
    if not path.exists():
        print(f"  [MISSING] {path}")
        return None
    data = dict(np.load(path, allow_pickle=True))
    n_ep = int(data["n_episodes"])
    total_t = data["skull_xpos"].shape[0]
    print(f"  [OK] {condition}: {n_ep} episodes, {total_t} total timesteps")
    return data


def segment_episodes(data):
    """Split flat concatenated arrays into per-episode lists."""
    episode_lengths = data["episode_lengths"]
    gap_episode_ids = data["gap_episode_ids"]
    n_episodes = int(data["n_episodes"])

    has_touch = "palm_l_touch" in data and data["palm_l_touch"] is not None

    episodes = []
    offset = 0
    for ep in range(n_episodes):
        ep_len = int(episode_lengths[ep])
        sl = slice(offset, offset + ep_len)
        gap_mask = gap_episode_ids == ep
        ep_dict = {
            "skull_xpos": data["skull_xpos"][sl],
            "skull_xmat": data["skull_xmat"][sl],
            "torso_xpos": data["torso_xpos"][sl],
            "torso_xmat": data["torso_xmat"][sl],
            "torso_linvel": data["torso_linvel"][sl],
            "hand_l_xpos": (data["hand_l_xpos"][sl] if "hand_l_xpos" in data else None),
            "hand_r_xpos": (data["hand_r_xpos"][sl] if "hand_r_xpos" in data else None),
            "palm_l_touch": data["palm_l_touch"][sl] if has_touch else None,
            "palm_r_touch": data["palm_r_touch"][sl] if has_touch else None,
            "gap_leading_edges": data["gap_leading_edges_flat"][gap_mask],
            "gap_lengths": data["gap_lengths_flat"][gap_mask],
        }
        episodes.append(ep_dict)
        offset += ep_len
    return episodes


# Use inline data if available, otherwise load from npz
print("Loading kinematics data...")
all_data = {}

if inline_data is not None:
    # Use inline rollout data
    all_data["binocular"] = inline_data
    n_ep = int(inline_data["n_episodes"])
    total_t = inline_data["skull_xpos"].shape[0]
    print(f"  [INLINE] binocular: {n_ep} episodes, {total_t} total timesteps")
    # Load monocular conditions from npz if available
    for cond in ["monocular_left", "monocular_right"]:
        all_data[cond] = load_kinematics_data(DATA_DIR, cond)
else:
    for cond in CONDITIONS:
        all_data[cond] = load_kinematics_data(DATA_DIR, cond)

has_data = any(v is not None for v in all_data.values())
if not has_data:
    print("\n⚠ No data found. Set GENERATE_DATA=True or run collect_run_gap_data.py.")

## 2b. Controlled Psychometric Evaluation

The original psychometric analysis (Section 2) suffers from **survivorship bias**: the agent only attempts gap N if it survived gaps 1 through N-1. This inflates observed success rates.

**Fix:** Run many episodes (200+ per condition) and analyze **gap 1 only** — every episode attempts gap 1, so there is zero selection bias. Gap 2 is analyzed conditionally (only among episodes that crossed gap 1).

For each episode we only need to record:
- Gap 1 (and 2, 3) distances (randomized per episode)
- Maximum forward progress (`max torso_x`)
- Success = `max_x > gap_far_edge + 1cm` (reached the next platform)

The logistic fit gives the **50% threshold** — the gap distance at which the agent succeeds half the time. Comparing thresholds across conditions (binocular vs. monocular) reveals whether depth perception matters.

In [ ]:
if GENERATE_DATA:
    import time

    # ── Config ──
    PSYCHO_N_ENVS = 1024
    PSYCHO_N_BATCHES = 2
    PSYCHO_MAX_STEPS = 250
    N_GAPS_TO_TRACK = 3

    total_eps = PSYCHO_N_ENVS * PSYCHO_N_BATCHES
    print(
        f"Psychometric eval: {total_eps} episodes/condition "
        f"({PSYCHO_N_BATCHES}x{PSYCHO_N_ENVS}), max {PSYCHO_MAX_STEPS} steps"
    )
    print(f"Conditions: {CONDITIONS}")
    print("Recording full kinematics for downstream analysis.\n")

    # ── Build nworld=1024 renderers ──
    env_cfg = ckpt_config["env_config"]
    _prk = dict(
        mj_model=base_env.mj_model,
        mjx_model=base_env.mjx_model,
        nworld=PSYCHO_N_ENVS,
        width=env_cfg.get("vision_width", 32),
        height=env_cfg.get("vision_height", 32),
        grayscale=env_cfg.get("grayscale", True),
        use_textures=env_cfg.get("use_textures", True),
        use_shadows=env_cfg.get("use_shadows", True),
    )
    psycho_left_renderer = JaxVisionRenderer(
        camera_name=env_cfg.get("left_camera_name", "eye_left-rodent"), **_prk
    )
    psycho_right_renderer = JaxVisionRenderer(
        camera_name=env_cfg.get("right_camera_name", "eye_right-rodent"), **_prk
    )

    # Resolve forelimb body IDs for contact-based jump distance
    hand_l_id = mj_model.body("hand_L-rodent").id
    hand_r_id = mj_model.body("hand_R-rodent").id

    # Resolve touch sensor addresses for physics-based contact detection
    palm_l_adr = mj_model.sensor("palm_L-rodent").id
    palm_r_adr = mj_model.sensor("palm_R-rodent").id
    palm_l_sensoradr = mj_model.sensor_adr[palm_l_adr]
    palm_r_sensoradr = mj_model.sensor_adr[palm_r_adr]

    _pv_reset = jax.vmap(wrapped_env.reset)
    _pv_step = jax.vmap(wrapped_env.step)

    def _p_render_inject(state):
        l = psycho_left_renderer.render(state.data)
        r = psycho_right_renderer.render(state.data)
        v = jnp.concatenate([l, r], axis=-1)
        return state.replace(obs=VisionRenderWrapper._inject_vision(state.obs, v))

    @jax.jit
    def psycho_reset(rngs):
        return _p_render_inject(_pv_reset(rngs))

    @jax.jit
    def psycho_step(state, actions):
        return _p_render_inject(_pv_step(state, actions))

    # ── Run evaluation with kinematics recording ──
    psycho = {}

    for condition in CONDITIONS:
        t0 = time.time()
        rng_p = jax.random.PRNGKey(SEED + sum(ord(c) for c in condition))

        # Per-condition episode storage
        ep_skull_xpos_all, ep_skull_xmat_all = [], []
        ep_torso_xpos_all, ep_torso_xmat_all, ep_torso_linvel_all = [], [], []
        ep_hand_l_xpos_all, ep_hand_r_xpos_all = [], []
        ep_palm_l_touch_all, ep_palm_r_touch_all = [], []
        ep_lengths_all = []
        ep_gap_edges_all, ep_gap_lens_all, ep_gap_ids_all = [], [], []

        for bi in range(PSYCHO_N_BATCHES):
            rng_p, batch_rng, act_rng = jax.random.split(rng_p, 3)
            reset_rngs = jax.random.split(batch_rng, PSYCHO_N_ENVS)

            state = psycho_reset(reset_rngs)
            g_edges, g_lens = _extract_gap_geometry_batch(base_env, state.data)

            # Pre-allocate batch arrays: (T, N_ENVS, ...)
            T = PSYCHO_MAX_STEPS + 1
            b_skull_xpos = np.zeros((T, PSYCHO_N_ENVS, 3), dtype=np.float32)
            b_skull_xmat = np.zeros((T, PSYCHO_N_ENVS, 3, 3), dtype=np.float32)
            b_torso_xpos = np.zeros((T, PSYCHO_N_ENVS, 3), dtype=np.float32)
            b_torso_xmat = np.zeros((T, PSYCHO_N_ENVS, 3, 3), dtype=np.float32)
            b_torso_lv = np.zeros((T, PSYCHO_N_ENVS, 3), dtype=np.float32)
            b_hand_l_xpos = np.zeros((T, PSYCHO_N_ENVS, 3), dtype=np.float32)
            b_hand_r_xpos = np.zeros((T, PSYCHO_N_ENVS, 3), dtype=np.float32)
            b_palm_l_touch = np.zeros((T, PSYCHO_N_ENVS), dtype=np.float32)
            b_palm_r_touch = np.zeros((T, PSYCHO_N_ENVS), dtype=np.float32)
            b_done = np.zeros((T, PSYCHO_N_ENVS), dtype=bool)

            # Record t=0
            b_skull_xpos[0] = np.asarray(state.data.xpos[:, skull_id])
            b_skull_xmat[0] = np.asarray(state.data.xmat[:, skull_id])
            b_torso_xpos[0] = np.asarray(state.data.xpos[:, torso_id])
            b_torso_xmat[0] = np.asarray(state.data.xmat[:, torso_id])
            b_torso_lv[0] = np.asarray(state.data._impl.subtree_linvel[:, torso_id])
            b_hand_l_xpos[0] = np.asarray(state.data.xpos[:, hand_l_id])
            b_hand_r_xpos[0] = np.asarray(state.data.xpos[:, hand_r_id])
            sdata_0 = np.asarray(state.data.sensordata)
            b_palm_l_touch[0] = sdata_0[:, palm_l_sensoradr]
            b_palm_r_touch[0] = sdata_0[:, palm_r_sensoradr]

            env_done = np.zeros(PSYCHO_N_ENVS, dtype=bool)
            actual_steps = PSYCHO_MAX_STEPS

            for t in range(PSYCHO_MAX_STEPS):
                obs = state.obs
                if condition != "binocular":
                    obs = {
                        **obs,
                        "vision": apply_monocular_mask(obs["vision"], condition),
                    }
                _, act_rng = jax.random.split(act_rng)
                actions, _ = policy_fn(params_tuple, obs, act_rng)
                state = psycho_step(state, actions)

                b_skull_xpos[t + 1] = np.asarray(state.data.xpos[:, skull_id])
                b_skull_xmat[t + 1] = np.asarray(state.data.xmat[:, skull_id])
                b_torso_xpos[t + 1] = np.asarray(state.data.xpos[:, torso_id])
                b_torso_xmat[t + 1] = np.asarray(state.data.xmat[:, torso_id])
                b_torso_lv[t + 1] = np.asarray(
                    state.data._impl.subtree_linvel[:, torso_id]
                )
                b_hand_l_xpos[t + 1] = np.asarray(state.data.xpos[:, hand_l_id])
                b_hand_r_xpos[t + 1] = np.asarray(state.data.xpos[:, hand_r_id])
                sdata_t = np.asarray(state.data.sensordata)
                b_palm_l_touch[t + 1] = sdata_t[:, palm_l_sensoradr]
                b_palm_r_touch[t + 1] = sdata_t[:, palm_r_sensoradr]

                step_done = np.asarray(state.done) > 0.5
                b_done[t + 1] = step_done
                env_done |= step_done
                if np.all(env_done):
                    actual_steps = t + 1
                    break

            # Segment per-episode and append
            ep_base = bi * PSYCHO_N_ENVS
            for ei in range(PSYCHO_N_ENVS):
                done_idx = np.where(b_done[1 : actual_steps + 1, ei])[0]
                ep_steps = int(done_idx[0]) + 1 if len(done_idx) > 0 else actual_steps
                ep_len = ep_steps + 1  # +1 for t=0

                ep_skull_xpos_all.append(b_skull_xpos[:ep_len, ei].copy())
                ep_skull_xmat_all.append(b_skull_xmat[:ep_len, ei].copy())
                ep_torso_xpos_all.append(b_torso_xpos[:ep_len, ei].copy())
                ep_torso_xmat_all.append(b_torso_xmat[:ep_len, ei].copy())
                ep_torso_linvel_all.append(b_torso_lv[:ep_len, ei].copy())
                ep_hand_l_xpos_all.append(b_hand_l_xpos[:ep_len, ei].copy())
                ep_hand_r_xpos_all.append(b_hand_r_xpos[:ep_len, ei].copy())
                ep_palm_l_touch_all.append(b_palm_l_touch[:ep_len, ei].copy())
                ep_palm_r_touch_all.append(b_palm_r_touch[:ep_len, ei].copy())
                ep_lengths_all.append(ep_len)

                ep_id = ep_base + ei
                ep_gap_edges_all.append(g_edges[ei, :N_GAPS_TO_TRACK])
                ep_gap_lens_all.append(g_lens[ei, :N_GAPS_TO_TRACK])
                ep_gap_ids_all.append(np.full(N_GAPS_TO_TRACK, ep_id, dtype=np.int32))

            print(
                f"  {condition} batch {bi+1}/{PSYCHO_N_BATCHES} done "
                f"(t={actual_steps} steps)"
            )

        # ── Pack into npz-compatible dict ──
        cond_data = {
            "skull_xpos": np.concatenate(ep_skull_xpos_all, axis=0),
            "skull_xmat": np.concatenate(ep_skull_xmat_all, axis=0),
            "torso_xpos": np.concatenate(ep_torso_xpos_all, axis=0),
            "torso_xmat": np.concatenate(ep_torso_xmat_all, axis=0),
            "torso_linvel": np.concatenate(ep_torso_linvel_all, axis=0),
            "hand_l_xpos": np.concatenate(ep_hand_l_xpos_all, axis=0),
            "hand_r_xpos": np.concatenate(ep_hand_r_xpos_all, axis=0),
            "palm_l_touch": np.concatenate(ep_palm_l_touch_all, axis=0),
            "palm_r_touch": np.concatenate(ep_palm_r_touch_all, axis=0),
            "episode_lengths": np.array(ep_lengths_all, dtype=np.int32),
            "n_episodes": total_eps,
            "gap_leading_edges_flat": np.concatenate(ep_gap_edges_all, axis=0),
            "gap_lengths_flat": np.concatenate(ep_gap_lens_all, axis=0),
            "gap_episode_ids": np.concatenate(ep_gap_ids_all, axis=0),
        }

        # Save to disk
        save_path = DATA_DIR / f"run_gap_kinematics_{condition}.npz"
        np.savez_compressed(
            save_path,
            **{
                k: np.array(v) if not isinstance(v, np.ndarray) else v
                for k, v in cond_data.items()
            },
        )
        total_ts = cond_data["skull_xpos"].shape[0]
        print(
            f"  Saved: {save_path.name} "
            f"({save_path.stat().st_size / 1024:.0f} KB, {total_ts} timesteps)"
        )

        # Update all_data for downstream analysis cells
        all_data[condition] = cond_data

        # Compute psychometric summary
        max_x = np.array([ep[:, 0].max() for ep in ep_torso_xpos_all])
        gap_dists = np.array(ep_gap_lens_all)  # (total_eps, N_GAPS_TO_TRACK)
        gap_starts = np.array(ep_gap_edges_all)  # (total_eps, N_GAPS_TO_TRACK)
        crossed = max_x[:, None] > (gap_starts + gap_dists) + 0.01

        psycho[condition] = {
            "gap_distances": gap_dists,
            "gap_starts": gap_starts,
            "max_x": max_x,
            "crossed": crossed,
        }

        elapsed = time.time() - t0
        print(
            f"{condition}: {elapsed:.1f}s, "
            f"gap-1={crossed[:, 0].mean():.1%}, "
            f"gap-2={crossed[:, 1].mean():.1%}\n"
        )

    # Update flags for downstream cells
    has_data = True

    # Clean up renderers
    del psycho_left_renderer, psycho_right_renderer, _pv_reset, _pv_step
    gc.collect()

    # ════════════════════════════════════════════════════════════
    # Plot psychometric curves
    # ════════════════════════════════════════════════════════════
    from scipy.optimize import curve_fit

    def logistic(x, x0, k):
        return 1.0 / (1.0 + np.exp(k * (x - x0)))

    def wilson_ci(n_succ, n_tot, z=1.96):
        if n_tot == 0:
            return 0, 0, 0
        p = n_succ / n_tot
        d = 1 + z**2 / n_tot
        center = (p + z**2 / (2 * n_tot)) / d
        margin = z * np.sqrt(p * (1 - p) / n_tot + z**2 / (4 * n_tot**2)) / d
        return center, max(0, center - margin), min(1, center + margin)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    gap_labels = ["Gap 1 (unbiased)", "Gap 2 (conditional on crossing gap 1)"]

    for gi, ax in enumerate(axes):
        ax.set_title(gap_labels[gi])
        for cond in CONDITIONS:
            if cond not in psycho:
                continue
            data = psycho[cond]
            dists = data["gap_distances"][:, gi]
            success = data["crossed"][:, gi]
            if gi > 0:
                reached = data["crossed"][:, :gi].all(axis=1)
                dists, success = dists[reached], success[reached]
                if len(dists) < 10:
                    continue

            n_bins = 8
            edges = np.linspace(dists.min() - 1e-6, dists.max() + 1e-6, n_bins + 1)
            bc, br, blo, bhi = [], [], [], []
            for b in range(n_bins):
                mask = (dists >= edges[b]) & (dists < edges[b + 1])
                nt = mask.sum()
                if nt < 3:
                    continue
                ns = success[mask].sum()
                center, lo, hi = wilson_ci(ns, nt)
                bc.append((edges[b] + edges[b + 1]) / 2)
                br.append(center)
                blo.append(lo)
                bhi.append(hi)
            if not bc:
                continue
            bc, br, blo, bhi = map(np.array, [bc, br, blo, bhi])
            color = COLORS.get(cond, "gray")
            ax.errorbar(
                bc * 100,
                br,
                yerr=[br - blo, bhi - br],
                fmt="o",
                color=color,
                capsize=3,
                markersize=5,
                label=f"{cond} (n={len(dists)})",
            )
            try:
                popt, _ = curve_fit(
                    logistic,
                    dists,
                    success.astype(float),
                    p0=[dists.mean(), 50],
                    maxfev=5000,
                )
                x_fit = np.linspace(dists.min(), dists.max(), 200)
                ax.plot(
                    x_fit * 100,
                    logistic(x_fit, *popt),
                    color=color,
                    linewidth=2,
                    alpha=0.6,
                )
                ax.axvline(
                    popt[0] * 100,
                    color=color,
                    ls=":",
                    alpha=0.4,
                    label=f"  50% @ {popt[0]*100:.1f}cm",
                )
            except (RuntimeError, ValueError):
                pass
        ax.set_xlabel("Gap distance (cm)")
        ax.set_ylabel("Success rate")
        ax.set_ylim(-0.05, 1.05)
        ax.axhline(0.5, color="gray", ls="--", alpha=0.3)
        ax.legend(fontsize=7, loc="lower left")

    plt.suptitle(
        f"Controlled Psychometric Curves ({total_eps} episodes/condition)",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "psychometric_controlled.png", dpi=150, bbox_inches="tight"
    )
    plt.show()

    # Summary
    print(f"\nSummary ({total_eps} episodes/condition):")
    for cond in CONDITIONS:
        if cond not in psycho:
            continue
        d = psycho[cond]
        for gi in range(min(2, d["crossed"].shape[1])):
            dists = d["gap_distances"][:, gi]
            succ = d["crossed"][:, gi]
            if gi > 0:
                reached = d["crossed"][:, :gi].all(axis=1)
                dists, succ = dists[reached], succ[reached]
            if len(dists) > 0:
                print(
                    f"  {cond} gap {gi+1}: {succ.mean():.1%} "
                    f"({succ.sum()}/{len(succ)}), "
                    f"mean dist={dists.mean()*100:.1f}cm"
                )
else:
    psycho = {}
    print("Skipping — GENERATE_DATA=False")

## 2c. Jump Distance vs Gap Distance

Does the virtual rodent calibrate its jump to the gap width? If it uses depth estimation to plan the jump, we expect:
- **Jump distance** (takeoff-to-landing) scales with gap distance
- **Peak forward velocity** increases for wider gaps (harder launch)
- **Jump height** increases for wider gaps (higher arc for longer flight)

A positive correlation between gap distance and these jump metrics indicates that the agent visually estimates gap width and modulates its motor output accordingly — not just executing a stereotyped jump.

In [ ]:
if has_data:
    import seaborn as sns

    # ── Contact-based jump distance for each gap crossing (binocular only) ──
    # Jump distance = x-distance between last forelimb contact on source
    # platform and first forelimb contact on landing platform.
    # Contact is detected via MuJoCo palm touch sensors (physics-based).
    # Falls back to z-threshold if touch sensor data is not available.

    CONTACT_Z_THRESH = 0.025  # fallback: 2.5cm above platform surface
    MARGIN = 0.015  # 1.5cm margin for platform edge detection
    JUMP_COND = "binocular"

    jumps = []  # list of dicts per gap crossing

    raw_data = all_data.get(JUMP_COND)
    has_hand_data = (
        raw_data is not None
        and "hand_l_xpos" in raw_data
        and raw_data["hand_l_xpos"] is not None
    )
    has_touch_data = (
        raw_data is not None
        and "palm_l_touch" in raw_data
        and raw_data["palm_l_touch"] is not None
    )

    if has_touch_data:
        print("Using physics-based palm touch sensors for contact detection.")
    elif has_hand_data:
        print("Touch sensor data not available; falling back to z-threshold heuristic.")

    if has_hand_data:
        episodes = segment_episodes(raw_data)

        for ep_idx, ep in enumerate(episodes):
            hand_l = ep["hand_l_xpos"]  # (T, 3)
            hand_r = ep["hand_r_xpos"]  # (T, 3)
            tx = ep["torso_xpos"][:, 0]  # forward position (for gap detection)

            # Use the lower (more forward) hand at each timestep
            hand_x = np.minimum(hand_l[:, 0], hand_r[:, 0])  # min x = trailing hand
            hand_x_max = np.maximum(hand_l[:, 0], hand_r[:, 0])  # max x = leading hand

            # Contact detection: prefer touch sensors, fall back to z-threshold
            if has_touch_data and ep["palm_l_touch"] is not None:
                palm_contact = (ep["palm_l_touch"] > 0) | (ep["palm_r_touch"] > 0)
            else:
                hand_z_min = np.minimum(hand_l[:, 2], hand_r[:, 2])
                palm_contact = hand_z_min < CONTACT_Z_THRESH

            for g_idx in range(len(ep["gap_leading_edges"])):
                gap_start = ep["gap_leading_edges"][g_idx]
                gap_len = ep["gap_lengths"][g_idx]
                gap_end = gap_start + gap_len

                # ── Takeoff: last frame where a forelimb is on the source platform ──
                # Hand is on source: hand_x < gap_start AND palm in contact
                on_source = (hand_x_max < gap_start) & palm_contact
                source_frames = np.where(on_source)[0]
                if len(source_frames) == 0:
                    continue
                t_takeoff = source_frames[-1]

                # Record takeoff hand x (use the more forward hand at takeoff)
                takeoff_hand_x = max(hand_l[t_takeoff, 0], hand_r[t_takeoff, 0])

                # ── Landing: first frame (after takeoff) where a forelimb
                #    is on the landing platform ──
                after_takeoff = np.arange(len(hand_x)) > t_takeoff
                on_landing = (hand_x > gap_end) & palm_contact & after_takeoff
                landing_frames = np.where(on_landing)[0]
                crossed = len(landing_frames) > 0

                if crossed:
                    t_land = landing_frames[0]
                    # Use the less forward hand at landing (first to touch)
                    land_hand_x = min(hand_l[t_land, 0], hand_r[t_land, 0])
                else:
                    continue  # skip failed crossings for contact measurement

                # ── Contact-based jump distance ──
                jump_distance = land_hand_x - takeoff_hand_x

                # Also compute velocity/height metrics from torso
                jump_slice = slice(t_takeoff, t_land + 1)
                torso_z = ep["torso_xpos"][jump_slice, 2]
                torso_x = ep["torso_xpos"][jump_slice, 0]
                if len(torso_x) < 2:
                    continue

                fwd_vel = np.diff(torso_x) / CTRL_DT
                peak_fwd_vel = fwd_vel.max() if len(fwd_vel) > 0 else 0.0
                z_at_takeoff = torso_z[0]
                peak_z = torso_z.max()

                jumps.append(
                    {
                        "ep": ep_idx,
                        "gap_idx": g_idx,
                        "gap_distance": gap_len,
                        "jump_distance": jump_distance,
                        "crossed": True,
                        "peak_fwd_velocity": peak_fwd_vel,
                        "height_gain": peak_z - z_at_takeoff,
                        "jump_duration": (t_land - t_takeoff) * CTRL_DT,
                        "takeoff_x": takeoff_hand_x,
                        "landing_x": land_hand_x,
                    }
                )

        print(f"Binocular contact-based crossings: {len(jumps)} successful")
    else:
        print(
            "Hand position data not available. Re-run psychometric eval "
            "(GENERATE_DATA=True) to collect forelimb kinematics."
        )

    if len(jumps) > 10:
        color = COLORS["binocular"]
        panel_size = 5  # each panel is 5x5 inches (square)
        fig, axes = plt.subplots(
            2, 2, figsize=(panel_size * 2 + 1, panel_size * 2 + 1), dpi=200
        )

        gd_all = np.array([j["gap_distance"] for j in jumps]) * 100
        jd_all = np.array([j["jump_distance"] for j in jumps]) * 100

        contact_method = "touch sensor" if has_touch_data else "z-threshold"

        # Shared x-axis limits based on actual gap distance range
        gap_xlim = (gd_all.min() - 0.5, gd_all.max() + 0.5)

        # ── Panel A: Contact jump distance vs gap distance (scatter) ──
        ax = axes[0, 0]
        ax.scatter(gd_all, jd_all, c=color, alpha=0.15, s=8)
        # Unity line spans the data range
        unity_range = [gap_xlim[0], max(gap_xlim[1], jd_all.max() + 0.5)]
        ax.plot(unity_range, unity_range, "k--", alpha=0.3, label="jump = gap")
        r, p = stats.pearsonr(gd_all / 100, jd_all / 100)
        ax.set_xlabel("Gap distance (cm)")
        ax.set_ylabel("Contact jump distance (cm)")
        ax.set_title(f"A. Jump vs Gap (r={r:.3f}, p={p:.1e})")
        ax.set_xlim(gap_xlim)
        ax.set_box_aspect(1)
        ax.legend(fontsize=8)
        sns.despine(ax=ax)

        # ── Panel B: Binned contact jump distance (mean + SEM) ──
        ax = axes[0, 1]
        n_bins = 6
        bin_edges = np.linspace(gd_all.min() - 0.1, gd_all.max() + 0.1, n_bins + 1)
        bx, by, be = [], [], []
        for b in range(n_bins):
            mask = (gd_all >= bin_edges[b]) & (gd_all < bin_edges[b + 1])
            if mask.sum() < 3:
                continue
            bx.append((bin_edges[b] + bin_edges[b + 1]) / 2)
            by.append(jd_all[mask].mean())
            be.append(stats.sem(jd_all[mask]))
        ax.errorbar(
            bx, by, yerr=be, fmt="o-", color=color, capsize=4, markersize=6, linewidth=2
        )
        # Unity line within the binned x range
        bx_range = [min(bx) - 0.5, max(bx) + 0.5]
        ax.plot(bx_range, bx_range, "k--", alpha=0.3, label="jump = gap")
        ax.set_xlabel("Gap distance (cm)")
        ax.set_ylabel("Mean contact jump distance (cm)")
        ax.set_title("B. Jump Scales with Gap (binned)")
        ax.set_xlim(gap_xlim)
        ax.set_box_aspect(1)
        ax.legend(fontsize=8)
        sns.despine(ax=ax)

        # ── Panel C: Peak forward velocity vs gap distance ──
        ax = axes[1, 0]
        pv = np.array([j["peak_fwd_velocity"] for j in jumps])
        bin_edges_m = np.linspace(gd_all.min() - 0.1, gd_all.max() + 0.1, 7)
        bx, by, be = [], [], []
        for b in range(6):
            mask = (gd_all >= bin_edges_m[b]) & (gd_all < bin_edges_m[b + 1])
            if mask.sum() < 3:
                continue
            bx.append((bin_edges_m[b] + bin_edges_m[b + 1]) / 2)
            by.append(pv[mask].mean())
            be.append(stats.sem(pv[mask]))
        ax.errorbar(
            bx, by, yerr=be, fmt="s-", color=color, capsize=4, markersize=6, linewidth=2
        )
        ax.set_xlabel("Gap distance (cm)")
        ax.set_ylabel("Peak forward velocity (m/s)")
        ax.set_title("C. Peak Velocity vs Gap Distance")
        ax.set_xlim(gap_xlim)
        ax.set_box_aspect(1)
        sns.despine(ax=ax)

        # ── Panel D: Height gain during jump ──
        ax = axes[1, 1]
        hg = np.array([j["height_gain"] for j in jumps]) * 1000
        bx, by, be = [], [], []
        for b in range(6):
            mask = (gd_all >= bin_edges_m[b]) & (gd_all < bin_edges_m[b + 1])
            if mask.sum() < 3:
                continue
            bx.append((bin_edges_m[b] + bin_edges_m[b + 1]) / 2)
            by.append(hg[mask].mean())
            be.append(stats.sem(hg[mask]))
        ax.errorbar(
            bx, by, yerr=be, fmt="^-", color=color, capsize=4, markersize=6, linewidth=2
        )
        ax.set_xlabel("Gap distance (cm)")
        ax.set_ylabel("Height gain at jump (mm)")
        ax.set_title("D. Jump Height vs Gap Distance")
        ax.set_xlim(gap_xlim)
        ax.set_box_aspect(1)
        sns.despine(ax=ax)

        plt.suptitle(
            f"Contact-Based Jump Calibration ({contact_method}): Forelimb Takeoff → Landing Distance",
            fontsize=14,
            fontweight="bold",
        )
        plt.tight_layout()

        # Save to checkpoint directory
        ckpt_figures_dir = CHECKPOINT_DIR / "figures"
        ckpt_figures_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            ckpt_figures_dir / "jump_distance_contact.png", dpi=300, bbox_inches="tight"
        )
        fig.savefig(
            FIGURES_DIR / "jump_distance_contact.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        print(f"Saved to {ckpt_figures_dir / 'jump_distance_contact.png'}")

        # ── Correlation stats ──
        gd_m = np.array([j["gap_distance"] for j in jumps])
        jd_m = np.array([j["jump_distance"] for j in jumps])
        pv_m = np.array([j["peak_fwd_velocity"] for j in jumps])
        hg_m = np.array([j["height_gain"] for j in jumps])
        r_jd, p_jd = stats.pearsonr(gd_m, jd_m)
        r_pv, p_pv = stats.pearsonr(gd_m, pv_m)
        r_hg, p_hg = stats.pearsonr(gd_m, hg_m)
        print(
            f"\nContact-based jump metrics ({len(jumps)} crossings, {contact_method}):"
        )
        print(f"  gap vs contact_jump_dist: r={r_jd:.3f}, p={p_jd:.2e}")
        print(f"  gap vs peak_velocity:     r={r_pv:.3f}, p={p_pv:.2e}")
        print(f"  gap vs height_gain:       r={r_hg:.3f}, p={p_hg:.2e}")
    else:
        print("Not enough gap crossings for analysis.")
else:
    print("Skipping — no data loaded.")

In [ ]:
# ── Recompute condition_metrics with full dataset ──
# Section 2 computed these before the psychometric eval populated monocular data.
# Now all_data has all 3 conditions, so recompute for Sections 3-6.

if has_data:
    condition_metrics = {}

    for cond, raw_data in all_data.items():
        if raw_data is None:
            continue

        episodes = segment_episodes(raw_data)

        all_windows = []
        all_skull_z = []
        all_pitch = []
        all_skull_pos = []
        all_torso_vel = []

        for ep in episodes:
            torso_x = ep["torso_xpos"][:, 0]
            windows = detect_approach_windows(
                torso_x=torso_x,
                gap_leading_edges=ep["gap_leading_edges"],
                gap_lengths=ep["gap_lengths"],
                window_steps=25,
            )
            for w in windows:
                idx = w.timestep_indices
                skull_z = ep["skull_xpos"][idx, 2]
                pitch_vals = []
                for t in idx:
                    pose = extract_head_pose(
                        skull_xpos=ep["skull_xpos"][t],
                        skull_xmat=ep["skull_xmat"][t],
                        torso_xmat=ep["torso_xmat"][t],
                    )
                    pitch_vals.append(pose.pitch_deg)

                all_windows.append(w)
                all_skull_z.append(skull_z)
                all_pitch.append(np.array(pitch_vals))
                all_skull_pos.append(ep["skull_xpos"][idx])
                all_torso_vel.append(np.diff(ep["torso_xpos"][idx], axis=0) / CTRL_DT)

        if all_windows:
            metrics = compute_gap_metrics(
                approach_windows=all_windows,
                skull_z_traces=all_skull_z,
                pitch_traces=all_pitch,
                skull_position_traces=all_skull_pos,
                torso_velocity_traces=all_torso_vel,
                dt=CTRL_DT,
            )
        else:
            metrics = []
        condition_metrics[cond] = metrics
        print(f"{cond}: {len(metrics)} approach windows from {len(episodes)} episodes")

    has_metrics = any(len(v) > 0 for v in condition_metrics.values())

    if has_metrics:
        comparison = compare_conditions(condition_metrics)
        print(
            f"\nConditions with metrics: {[c for c, m in condition_metrics.items() if m]}"
        )
    else:
        comparison = {}
        print("\nNo approach windows detected.")
else:
    condition_metrics = {}
    comparison = {}
    has_metrics = False

## 3. Motion Parallax Evidence (Parker et al. Figure 3D-E)

Do monocular agents increase vertical head movements? Motion parallax = using self-generated motion to create depth cues from a single viewpoint.

In [ ]:
if has_metrics:
    # Bar chart: vertical movement count and amplitude
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    for i, cond in enumerate(CONDITIONS):
        if cond not in condition_metrics or not condition_metrics[cond]:
            continue
        counts = [m["n_vertical_movements"] for m in condition_metrics[cond]]
        ax.bar(
            i, np.mean(counts), yerr=stats.sem(counts), color=COLORS[cond], capsize=5
        )
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, rotation=20, ha="right")
    ax.set_ylabel("Vertical movements per approach")
    ax.set_title("3D: Vertical Head Movement Count")

    ax = axes[1]
    for i, cond in enumerate(CONDITIONS):
        if cond not in condition_metrics or not condition_metrics[cond]:
            continue
        amps = [m["movement_amplitude"] for m in condition_metrics[cond]]
        ax.bar(
            i,
            np.mean(amps) * 1000,
            yerr=stats.sem(amps) * 1000,
            color=COLORS[cond],
            capsize=5,
        )
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, rotation=20, ha="right")
    ax.set_ylabel("Movement amplitude (mm)")
    ax.set_title("3E: Vertical Head Movement Amplitude")

    plt.suptitle("Motion Parallax: Vertical Head Movements", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "motion_parallax_vertical.png", dpi=150)
    plt.show()

    # Use module plotting: metric vs gap distance
    if comparison:
        fig = plot_metric_vs_gap_distance(
            comparison, "n_vertical_movements", "Vertical movements"
        )
        fig.savefig(FIGURES_DIR / "vert_movements_vs_gap.png", dpi=150)
        plt.show()

    # Stats: binocular vs monocular
    if "binocular" in condition_metrics and condition_metrics["binocular"]:
        bino_counts = [
            m["n_vertical_movements"] for m in condition_metrics["binocular"]
        ]
        for mono in ["monocular_left", "monocular_right"]:
            if mono in condition_metrics and condition_metrics[mono]:
                mono_counts = [
                    m["n_vertical_movements"] for m in condition_metrics[mono]
                ]
                u_stat, p_val = stats.mannwhitneyu(
                    bino_counts, mono_counts, alternative="two-sided"
                )
                print(
                    f"Binocular vs {mono} (vert count): U={u_stat:.0f}, p={p_val:.4f}"
                )
else:
    print("Skipping — no approach windows detected.")

## 4. Position Parallax Evidence (Parker et al. Figure 3I)

Does head pitch change under monocular conditions? Real mice pitch their heads down more when monocular, possibly to use the binocular zone of the remaining eye for depth estimation.

In [ ]:
if has_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Mean head pitch per condition
    ax = axes[0]
    pitch_data = {}
    for i, cond in enumerate(CONDITIONS):
        if cond not in condition_metrics or not condition_metrics[cond]:
            continue
        pitches = [m["mean_pitch"] for m in condition_metrics[cond]]
        pitch_data[cond] = pitches
        ax.bar(
            i, np.mean(pitches), yerr=stats.sem(pitches), color=COLORS[cond], capsize=5
        )
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, rotation=20, ha="right")
    ax.set_ylabel("Mean head pitch (deg)")
    ax.set_title("3I: Head Pitch During Approach")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

    # Pitch distribution (box plot)
    ax = axes[1]
    box_data = [pitch_data.get(c, []) for c in CONDITIONS if c in pitch_data]
    box_labels = [c for c in CONDITIONS if c in pitch_data]
    if box_data:
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch, cond in zip(bp["boxes"], box_labels):
            patch.set_facecolor(COLORS[cond])
            patch.set_alpha(0.6)
    ax.set_ylabel("Head pitch (deg)")
    ax.set_title("Head Pitch Distribution")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

    plt.suptitle("Position Parallax: Head Pitch", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "position_parallax_pitch.png", dpi=150)
    plt.show()

    # Stats
    if "binocular" in pitch_data:
        for mono in ["monocular_left", "monocular_right"]:
            if mono in pitch_data:
                t_stat, p_val = stats.ttest_ind(
                    pitch_data["binocular"], pitch_data[mono]
                )
                print(f"Binocular vs {mono} (pitch): t={t_stat:.2f}, p={p_val:.4f}")
else:
    print("Skipping — no approach windows detected.")

## 5. Temporal Integration (Parker et al. Figure 3G-H)

Do monocular agents take longer to approach or travel more total head distance? Parker et al. found monocular mice increase total head distance and approach duration.

In [ ]:
if has_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # 3G: Approach duration
    ax = axes[0]
    for i, cond in enumerate(CONDITIONS):
        if cond not in condition_metrics or not condition_metrics[cond]:
            continue
        durations = [m["approach_duration"] for m in condition_metrics[cond]]
        ax.bar(
            i,
            np.mean(durations) * 1000,
            yerr=stats.sem(durations) * 1000,
            color=COLORS[cond],
            capsize=5,
        )
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, rotation=20, ha="right")
    ax.set_ylabel("Approach duration (ms)")
    ax.set_title("3G: Approach Duration")

    # 3H: Total head distance
    ax = axes[1]
    for i, cond in enumerate(CONDITIONS):
        if cond not in condition_metrics or not condition_metrics[cond]:
            continue
        dists = [m["total_head_distance"] for m in condition_metrics[cond]]
        ax.bar(
            i,
            np.mean(dists) * 1000,
            yerr=stats.sem(dists) * 1000,
            color=COLORS[cond],
            capsize=5,
        )
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, rotation=20, ha="right")
    ax.set_ylabel("Total head distance (mm)")
    ax.set_title("3H: Total Head Distance During Approach")

    plt.suptitle("Temporal Integration Metrics", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "temporal_integration.png", dpi=150)
    plt.show()
else:
    print("Skipping — no approach windows detected.")

## 5b. Approach Angle Analysis: Oblique Approach Strategy

The virtual rodent appears to approach gaps at an **oblique angle** to the gap edge, then straighten to jump **perpendicular** to the edge. This is potentially an active sensing strategy:

- **Oblique approach** creates differential optical flow between the two eyes and across the visual field, enhancing depth estimation from motion parallax
- **Perpendicular takeoff** optimizes jump biomechanics (shortest gap crossing distance)

This behavior is predicted to be **more pronounced with textured environments** (outdoor_natural) because motion parallax requires visual texture to generate informative optical flow. Plain surfaces provide no texture gradients, so an angled approach yields no additional depth information.

**Metrics:**
- **Approach angle**: mean |heading deviation from corridor axis| during the early phase (first 60% of the 800ms approach window)
- **Takeoff angle**: same metric during the late phase (last 20%, right before gap crossing)
- **Angle reduction**: approach - takeoff (positive = straightening before jump)

In [ ]:
if has_data:
    from scipy.ndimage import uniform_filter1d

    # ── Settings ──
    APPROACH_STEPS = 80  # steps before gap crossing to analyze (800ms at 100Hz)
    VEL_SMOOTH = 7  # smoothing window for velocity heading

    def _heading_from_positions(xpos_xy, smooth=VEL_SMOOTH):
        """Heading angle (deg) in XY plane. 0° = along +X (corridor/gap normal)."""
        vel = np.gradient(xpos_xy, axis=0)
        if smooth > 1:
            vel = uniform_filter1d(vel, smooth, axis=0)
        return np.degrees(np.arctan2(vel[:, 1], vel[:, 0]))

    # ── Collect per-gap-crossing data ──
    crossings = []

    for cond, raw_data in all_data.items():
        if raw_data is None:
            continue
        episodes = segment_episodes(raw_data)

        for ep_idx, ep in enumerate(episodes):
            torso_xy = ep["torso_xpos"][:, :2]
            torso_x = torso_xy[:, 0]
            heading = _heading_from_positions(torso_xy)

            for g_idx in range(len(ep["gap_leading_edges"])):
                gap_x = ep["gap_leading_edges"][g_idx]
                gap_len = ep["gap_lengths"][g_idx]

                # Find first timestep where torso crosses gap leading edge
                cross_mask = torso_x >= gap_x
                if not cross_mask.any():
                    continue
                cross_t = int(np.argmax(cross_mask))

                # Extract approach window
                start_t = max(0, cross_t - APPROACH_STEPS)
                window_len = cross_t - start_t
                if window_len < 15:
                    continue

                win_heading = heading[start_t:cross_t]
                # Include some post-crossing steps for trajectory plot
                post = min(20, len(torso_xy) - cross_t)
                win_xy = torso_xy[start_t : cross_t + post].copy()

                # Speed filter: skip if agent is barely moving
                speed = np.linalg.norm(
                    np.gradient(torso_xy[start_t:cross_t], axis=0), axis=1
                )
                moving = speed > 0.0005
                if moving.sum() < 10:
                    continue

                n = len(win_heading)

                # Approach phase: first 60% of window (well before gap)
                early_end = int(n * 0.6)
                early_mov = moving[:early_end]
                approach_angle = (
                    float(np.mean(np.abs(win_heading[:early_end][early_mov])))
                    if early_mov.any()
                    else np.nan
                )

                # Takeoff phase: last 20% of window (right before crossing)
                late_start = int(n * 0.8)
                late_mov = moving[late_start:]
                takeoff_angle = (
                    float(np.mean(np.abs(win_heading[late_start:][late_mov])))
                    if late_mov.any()
                    else np.nan
                )

                crossings.append(
                    {
                        "cond": cond,
                        "ep": ep_idx,
                        "gap_idx": g_idx,
                        "gap_len": gap_len,
                        "gap_x": gap_x,
                        "approach_angle": approach_angle,
                        "takeoff_angle": takeoff_angle,
                        "heading_abs": np.abs(win_heading),
                        "t_rel_ms": np.arange(-window_len, 0) * CTRL_DT * 1000,
                        "xy": win_xy,
                        "cross_idx": cross_t - start_t,
                    }
                )

    n_crossings = len(crossings)
    print(f"Found {n_crossings} gap crossings across all conditions")

    if n_crossings > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # ── Panel A: Top-down XY trajectories near gaps ──
        ax = axes[0]
        for c in crossings[:50]:
            xy = c["xy"].copy()
            xy[:, 0] -= c["gap_x"]  # shift so gap edge = x=0
            ci = c["cross_idx"]
            color = COLORS.get(c["cond"], "gray")
            # Pre-crossing approach
            ax.plot(xy[:ci, 0], xy[:ci, 1], color=color, alpha=0.3, linewidth=0.8)
            # Post-crossing (dashed)
            ax.plot(
                xy[ci:, 0],
                xy[ci:, 1],
                color=color,
                alpha=0.15,
                linewidth=0.6,
                linestyle="--",
            )
            # Arrow at crossing point
            if 1 < ci < len(xy):
                dx = xy[ci, 0] - xy[ci - 2, 0]
                dy = xy[ci, 1] - xy[ci - 2, 1]
                scale = 0.01 / (np.hypot(dx, dy) + 1e-9)
                ax.annotate(
                    "",
                    xy=(xy[ci, 0], xy[ci, 1]),
                    xytext=(xy[ci, 0] - dx * scale, xy[ci, 1] - dy * scale),
                    arrowprops=dict(arrowstyle="->", color=color, alpha=0.5, lw=1),
                )
        ax.axvline(0, color="red", ls="--", alpha=0.6, label="Gap leading edge")
        ax.set_xlabel("X relative to gap edge (m)")
        ax.set_ylabel("Y position (m)")
        ax.set_title("A. Top-Down Approach Trajectories")
        ax.legend(fontsize=8)

        # ── Panel B: |Heading angle| vs time-to-crossing ──
        ax = axes[1]
        t_grid = np.linspace(-APPROACH_STEPS * CTRL_DT * 1000, 0, 60)

        for cond in CONDITIONS:
            cond_traces = [c for c in crossings if c["cond"] == cond]
            if not cond_traces:
                continue
            interp_arr = []
            for c in cond_traces:
                if len(c["t_rel_ms"]) >= 2:
                    interp = np.interp(
                        t_grid,
                        c["t_rel_ms"],
                        c["heading_abs"],
                        left=np.nan,
                        right=np.nan,
                    )
                    interp_arr.append(interp)
            if not interp_arr:
                continue
            arr = np.array(interp_arr)
            n_valid = np.sum(~np.isnan(arr), axis=0).clip(1)
            mean_a = np.nanmean(arr, axis=0)
            sem_a = np.nanstd(arr, axis=0) / np.sqrt(n_valid)
            color = COLORS.get(cond, "gray")
            ax.plot(t_grid, mean_a, color=color, linewidth=2, label=cond)
            ax.fill_between(
                t_grid,
                mean_a - sem_a,
                mean_a + sem_a,
                color=color,
                alpha=0.15,
            )

        ax.set_xlabel("Time to gap crossing (ms)")
        ax.set_ylabel("|Heading angle| from corridor axis (deg)")
        ax.set_title("B. Heading Angle During Approach")
        ax.legend(fontsize=8)
        ax.axhline(0, color="gray", ls="--", alpha=0.3)

        # ── Panel C: Approach vs takeoff angle (paired) ──
        ax = axes[2]
        cond_present = [
            c for c in CONDITIONS if any(cr["cond"] == c for cr in crossings)
        ]
        for i, cond in enumerate(cond_present):
            cond_c = [
                c
                for c in crossings
                if c["cond"] == cond
                and not np.isnan(c["approach_angle"])
                and not np.isnan(c["takeoff_angle"])
            ]
            if not cond_c:
                continue
            approaches = np.array([c["approach_angle"] for c in cond_c])
            takeoffs = np.array([c["takeoff_angle"] for c in cond_c])
            color = COLORS.get(cond, "gray")
            x_a, x_t = i * 2.5, i * 2.5 + 1

            ax.bar(
                x_a,
                approaches.mean(),
                yerr=stats.sem(approaches),
                color=color,
                alpha=0.8,
                capsize=4,
                width=0.9,
            )
            ax.bar(
                x_t,
                takeoffs.mean(),
                yerr=stats.sem(takeoffs),
                color=color,
                alpha=0.4,
                capsize=4,
                width=0.9,
                hatch="//",
            )

            # Paired Wilcoxon signed-rank test
            n_pairs = len(approaches)
            if n_pairs >= 10:
                try:
                    w_stat, p_val = stats.wilcoxon(approaches, takeoffs)
                    sig = (
                        "***"
                        if p_val < 0.001
                        else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    )
                    y_top = (
                        max(approaches.mean(), takeoffs.mean())
                        + max(stats.sem(approaches), stats.sem(takeoffs))
                        + 1.5
                    )
                    ax.plot(
                        [x_a, x_a, x_t, x_t],
                        [y_top, y_top + 0.5, y_top + 0.5, y_top],
                        color="black",
                        linewidth=0.8,
                    )
                    ax.text(
                        (x_a + x_t) / 2,
                        y_top + 0.7,
                        sig,
                        ha="center",
                        fontsize=10,
                    )
                    print(
                        f"{cond}: approach={approaches.mean():.1f}"
                        f"\u00b1{stats.sem(approaches):.1f}\u00b0, "
                        f"takeoff={takeoffs.mean():.1f}"
                        f"\u00b1{stats.sem(takeoffs):.1f}\u00b0, "
                        f"Wilcoxon W={w_stat:.0f}, p={p_val:.4f} ({sig})"
                    )
                except ValueError:
                    pass  # all differences zero

        from matplotlib.patches import Patch

        ax.legend(
            handles=[
                Patch(facecolor="gray", alpha=0.8, label="Approach (early 60%)"),
                Patch(
                    facecolor="gray", alpha=0.4, hatch="//", label="Takeoff (last 20%)"
                ),
            ],
            fontsize=8,
        )
        ax.set_ylabel("|Heading angle| (deg)")
        ax.set_title("C. Approach vs. Takeoff Angle")
        tick_pos = [i * 2.5 + 0.5 for i in range(len(cond_present))]
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(cond_present, rotation=20, ha="right", fontsize=9)

        plt.suptitle(
            "Gap Approach Angle Analysis: Oblique Approach \u2192 Perpendicular Jump?",
            fontsize=14,
            fontweight="bold",
        )
        plt.tight_layout()
        plt.savefig(
            FIGURES_DIR / "approach_angle_analysis.png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.show()

        # ── Summary statistics ──
        for cond in cond_present:
            cond_c = [
                c
                for c in crossings
                if c["cond"] == cond
                and not np.isnan(c["approach_angle"])
                and not np.isnan(c["takeoff_angle"])
            ]
            if cond_c:
                a = np.array([c["approach_angle"] for c in cond_c])
                t = np.array([c["takeoff_angle"] for c in cond_c])
                reduction = a - t
                print(
                    f"{cond} ({len(cond_c)} crossings): "
                    f"mean angle reduction = {reduction.mean():.1f}\u00b0 "
                    f"\u00b1 {stats.sem(reduction):.1f}\u00b0"
                )
    else:
        print("No gap crossings found.")
else:
    print("Skipping \u2014 no data loaded.")

## 6. Combined Parker et al. Comparison Panel

2×4 multi-panel figure replicating the key Parker et al. analyses. Metrics plotted both as bar charts (across conditions) and as functions of gap distance.

In [ ]:
if has_metrics and comparison:
    fig = plot_parker_comparison_panel(comparison)
    fig.savefig(FIGURES_DIR / "parker_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    for metric, ylabel in [
        ("movement_amplitude", "Amplitude (m)"),
        ("mean_pitch", "Head pitch (deg)"),
        ("total_head_distance", "Head distance (m)"),
        ("mean_forward_velocity", "Forward velocity (m/s)"),
    ]:
        fig = plot_metric_vs_gap_distance(comparison, metric, ylabel)
        fig.savefig(FIGURES_DIR / f"{metric}_vs_gap.png", dpi=150)
        plt.show()
else:
    print("Skipping — no approach windows detected.")

## 7. Track 2: Single-Gap Psychometric Evaluation

Controlled single-gap testing using `run_gap_to_trial_eval.py` with fixed gap distances.

In [ ]:
single_gap_path = DATA_DIR / "single_gap" / "single_gap_psychometric.json"
if single_gap_path.exists():
    with open(single_gap_path) as f:
        sg_results = json.load(f)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    for cond, curves in sg_results.get("psychometric_curves", {}).items():
        dists = curves["distances"]
        rates = curves["success_rates"]
        ax.plot(
            dists,
            rates,
            marker="o",
            color=COLORS.get(cond, "gray"),
            label=cond,
            linewidth=2,
            markersize=8,
        )
    ax.set_xlabel("Gap distance (m)")
    ax.set_ylabel("Success rate")
    ax.set_title("Track 2: Single-Gap Psychometric Curve")
    ax.legend()
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "single_gap_psychometric.png", dpi=150)
    plt.show()

    print(f"Trials per distance: {sg_results.get('n_trials_per_distance', '?')}")
    print(f"Conditions tested: {sg_results.get('conditions', '?')}")
else:
    print(f"No single-gap data found at {single_gap_path}")
    print(
        "Run: python -m vnl_playground.tasks.rodent.analysis.run_gap_to_trial_eval \\"
    )
    print(
        "    --checkpoint_path /home/scott/SalkResearch/data/bino_run_gaps/260306_221728 \\"
    )
    print(
        "    --n_trials_per_distance 100 --output_dir ./outputs/motion_parallax/single_gap"
    )

## 8. Summary & Interpretation

**Key questions:**

1. **Psychometric curves**: Does the virtual rodent show decreasing success with increasing gap distance?
2. **Monocular robustness**: Does monocular masking impair performance, or is the agent robust like real mice?
3. **Motion parallax**: If robust under monocular conditions — does the agent increase vertical head movements?
4. **Position parallax**: Does head pitch change under monocular conditions?
5. **Temporal integration**: Does the agent spend more time / travel more distance approaching under monocular conditions?

**If motion parallax emerges naturally**, this is a strong result: the RL-trained virtual rodent, without any explicit reward for active sensing, develops the same compensatory strategy as real mice.

**If it does NOT emerge**, this is also interesting: it may suggest (a) the virtual rodent relies on different monocular cues (retinal image size, looming), (b) the camera model doesn't capture the optical properties needed for motion parallax, or (c) the reward structure doesn't incentivize active sensing.